# Hour 2 · The genre map — does the corpus sort itself?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexsosn/ugarit-dh-workshop/blob/main/notebooks/2b_similarity_clustering.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/alexsosn/ugarit-dh-workshop/main?labpath=notebooks%2F2b_similarity_clustering.ipynb)


**The headline demo.** We turn every tablet into a vector of its vocabulary, project all of them onto a 2-D map, and colour by genre. The question: **does the machine rediscover the genres philologists already know — without being told them?**

Hover any point to read that tablet. **Needs:** `scikit-learn`, `plotly`, `umap-learn` (all in `requirements.txt`; a PCA fallback runs if UMAP is missing).

*Reading:* `docs/04-genres.md`.

## 0. Setup


In [1]:
# === SETUP — run me first (works in Google Colab, Binder, and locally) ===
import os, sys, subprocess, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")

if "google.colab" in sys.modules:                      # we're on Colab
    REPO_URL = "https://github.com/alexsosn/ugarit-dh-workshop.git"
    REPO_DIR = "/content/ugarit-dh-workshop"
    if not os.path.isdir(REPO_DIR):                    # clone the repo once
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(os.path.join(REPO_DIR, "notebooks"))      # work from notebooks/
    # Colab already ships numpy/pandas/scikit-learn/matplotlib/plotly/networkx;
    # only umap-learn is usually missing. Install just that (fast).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "umap-learn"], check=False)

# make data/loader.py importable (we run from the notebooks/ folder)
sys.path.insert(0, os.path.abspath(".."))
from data.loader import load_texts

texts = load_texts()     # 1st Colab run downloads the CUC JSONL from HuggingFace, then caches it
print(f"Loaded {len(texts)} tablets — genre of the first one: {texts[0]['genre']}")

[loader] Loaded 278 CUC tablets, 25477 word tokens (source: AlexWalhai/cuc JSONL cache, licence: CC BY-NC 4.0).
Loaded 278 tablets — genre of the first one: myth


## 1. Turn each tablet into a TF-IDF vector
Each tablet becomes a point in a high-dimensional 'vocabulary space'.

In [2]:
sample = [t for t in texts if t["genre"] in {"myth","letter","ritual","divination"}
          and len(t["tokens"]) >= 30]
genres = [t["genre"] for t in sample]
from data.loader import corpus_as_documents
labels, docs = corpus_as_documents(sample)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs)
print(f"{X.shape[0]} tablets in a {X.shape[1]}-word space")

85 tablets in a 3198-word space


### Genre balance of the mapped tablets

Before the map: *how many* tablets of each genre are we actually projecting? The classes are **imbalanced** (letters dominate) — worth remembering when reading how tight or sparse each colour's cluster looks.

In [ ]:
import pandas as pd, plotly.express as px
from collections import Counter
gd = pd.Series(Counter(genres)).sort_values()
fig = px.bar(x=gd.values, y=gd.index, orientation="h", text=gd.values,
             color=gd.index, title="Genre balance of the mapped tablets")
fig.update_traces(textposition="outside")
fig.update_layout(height=300, width=640, showlegend=False,
                  xaxis_title="tablets", yaxis_title="",
                  margin=dict(l=120, r=30, t=50, b=30),
                  paper_bgcolor="white", plot_bgcolor="white")
fig.show()

## 2. Squash that space down to 2 dimensions
**UMAP** keeps similar tablets near each other. (If UMAP isn't installed, we fall back to PCA so the cell always runs.)

In [3]:
import numpy as np
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
try:
    import umap
    xy = umap.UMAP(n_components=2, n_neighbors=12, min_dist=0.25,
                   metric="cosine", random_state=42).fit_transform(X.toarray())
    METHOD = "UMAP"
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(X.toarray())
    METHOD = f"PCA (UMAP unavailable: {type(e).__name__})"
print("projected with", METHOD)

/Users/alexandersosnovschenko/projects/summer_school_2026/Antiquity Studies Summer School/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


projected with UMAP


## 3. The map — hover a point to read the tablet  ⭐
Colours are the *known* genres. Do they land in separate regions on their own?

In [4]:
import pandas as pd
import plotly.express as px

def preview(t, n=3):
    return "<br>".join(f"{r}: {l}" for r, l in zip(t["refs"][:n], t["lines"][:n]))

df = pd.DataFrame({
    "x": xy[:,0], "y": xy[:,1],
    "KTU": [t["ktu"] for t in sample], "name": [t["name"] for t in sample], "genre": genres,
    "words": [len(t["tokens"]) for t in sample],
    "text": [preview(t) for t in sample]})

fig = px.scatter(df, x="x", y="y", color="genre", hover_name="KTU",
                 hover_data={"x": False, "y": False, "name": True, "genre": True,
                             "words": True, "text": True},
                 title=f"CUC tablets in vocabulary space ({METHOD}) — hover to read")
fig.update_traces(marker=dict(size=12, opacity=0.85, line=dict(width=0.5, color="white")))
fig.update_layout(height=640, legend_title_text="genre",
                  xaxis_title="", yaxis_title="")
fig.show()

## 4. Which tablets are most alike?
The nearest neighbours of one tablet, by shared vocabulary.

In [5]:
from sklearn.metrics.pairwise import cosine_similarity
S = pd.DataFrame(cosine_similarity(X), index=labels, columns=labels)
TARGET = "1.4"   # a Baal-cycle tablet
print(f"Closest to KTU {TARGET}:")
print(S[TARGET].drop(TARGET).sort_values(ascending=False).head())

Closest to KTU 1.4:
1.3    0.639610
1.6    0.580772
1.5    0.513073
1.1    0.453415
1.2    0.393525
Name: 1.4, dtype: float64


## 5. Let the machine group them blind
`KMeans` clusters with **no** genre labels — then we check the clusters against the genres.

In [6]:
from sklearn.cluster import KMeans
km = KMeans(n_clusters=4, random_state=0, n_init=10).fit(X)
grid = pd.crosstab(pd.Series(km.labels_, name="cluster"), pd.Series(genres, name="genre"))
grid

genre,divination,letter,myth,ritual
cluster,,,,
0,2,4,12,2
1,1,0,0,13
2,2,35,0,0
3,0,14,0,0


## 6. Discussion
- Do letters land together? Do myths separate from rituals? Where does the machine **agree** with philologists, and where does it **disagree** (and is the disagreement interesting)?
- This is the whole workshop in one picture: raw text → vectors → a map that *means* something.

> **Production version:** UgaritLab serves a precomputed UMAP; here you ran a **live, interactive** one over the whole sample — and can reshape it yourself below.

## ✍️ Your turn
Edit one value below and re-run. Nothing here can break the notebook — if it goes sideways, just re-run the setup cell.


In [7]:
# Re-run sections 1–3 after changing either of these.
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")
MY_GENRES = {"myth", "letter", "ritual", "divination", "god-list"}  # ← add/remove a genre
NEIGHBORS = 8        # ← UMAP n_neighbors: smaller = more local clumps, larger = smoother

sample = [t for t in texts if t["genre"] in MY_GENRES and len(t["tokens"]) >= 30]
genres = [t["genre"] for t in sample]
labels, docs = corpus_as_documents(sample)
X = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs)
try:
    import umap
    xy = umap.UMAP(n_components=2, n_neighbors=NEIGHBORS, min_dist=0.25,
                   metric="cosine", random_state=42).fit_transform(X.toarray())
    METHOD = "UMAP"
except Exception as e:
    from sklearn.decomposition import PCA
    xy = PCA(n_components=2, random_state=42).fit_transform(X.toarray()); METHOD = "PCA"
print(f"{len(sample)} tablets — now re-run the map cell (section 3) to see them.")
# Hint: add "god-list" or "epic" and watch whether they carve out their own corner.

88 tablets — now re-run the map cell (section 3) to see them.


### ✍️ Your turn — make it 3-D

Two dimensions force the map to flatten things that may really sit apart. **Add a third axis** and rotate it with the mouse — genres that overlap in 2-D sometimes peel apart in 3-D (and sometimes the extra axis just spreads the same blob). This reuses your `MY_GENRES` / `NEIGHBORS` from the cell above, so tweak those and re-run.

**Spin it and ask:**
- Do **letters** stay one tight ball while **myth / ritual / divination** stretch along their own directions?
- Does the 3rd axis *reveal* a real split, or merely *inflate* the 2-D picture? Compare with the map in §3.
- Trust structure that survives **2-D, 3-D, and several `NEIGHBORS` values** — if a cluster only shows up once, be suspicious.

In [8]:
# === ✍️ Your turn — the 3-D map (drag to rotate) ===
import pandas as pd, plotly.express as px
warnings.filterwarnings("ignore", message="FNV hashing is not implemented in Numba.*")
warnings.filterwarnings("ignore", message="n_jobs value 1 overridden.*")

sample3 = [t for t in texts if t["genre"] in MY_GENRES and len(t["tokens"]) >= 30]
genres3 = [t["genre"] for t in sample3]
labels3, docs3 = corpus_as_documents(sample3)
X3 = TfidfVectorizer(token_pattern=r"[^\s]+").fit_transform(docs3)

try:
    import umap
    xyz = umap.UMAP(n_components=3, n_neighbors=NEIGHBORS, min_dist=0.25,
                    metric="cosine", random_state=42).fit_transform(X3.toarray())
    METHOD3 = "UMAP-3D"
except Exception as e:
    from sklearn.decomposition import PCA
    xyz = PCA(n_components=3, random_state=42).fit_transform(X3.toarray()); METHOD3 = "PCA-3D"

df3 = pd.DataFrame({"x": xyz[:, 0], "y": xyz[:, 1], "z": xyz[:, 2],
                    "KTU": [t["ktu"] for t in sample3], "name": [t["name"] for t in sample3], "genre": genres3,
                    "words": [len(t["tokens"]) for t in sample3],
                    "text": [preview(t) for t in sample3]})
fig = px.scatter_3d(df3, x="x", y="y", z="z", color="genre", hover_name="KTU",
                    hover_data={"x": False, "y": False, "z": False,
                                "name": True, "genre": True, "words": True, "text": True},
                    title=f"CUC tablets in 3-D vocabulary space ({METHOD3}) — drag to rotate")
fig.update_traces(marker=dict(size=5, opacity=0.85, line=dict(width=0.5, color="white")))
fig.update_layout(height=720, legend_title_text="genre",
                  scene=dict(xaxis_title="", yaxis_title="", zaxis_title=""))
fig.show()

### 🔭 Going further — more to extract from this map

The genre map is the doorway, not the whole house. Each idea below is a small add-on:

1. **Score it, don't eyeball it.** Replace "do the clusters match the genres?" with a number:
   `adjusted_rand_score(genres, km.labels_)` and `silhouette_score(X, km.labels_)` (from `sklearn.metrics`).
   On the default sample the ARI is ~0.5 — the *blind* clusters **partly** rediscover the philologists' genres.
2. **Why does a cluster cluster?** Print each KMeans cluster's **top TF-IDF words** (highest mean weight).
   Expect letters bound by address formulas (*l, rgm, thm, yslm*) and myth by divine names — the *vocabulary*
   behind the geometry.
3. **Who's misfiled?** For every tablet, find its **nearest genre centroid**; the tablets whose nearest
   centroid ≠ their catalogued genre are the machine's "I'd relabel this" list — hybrids, mixed tablets, or
   genuine miscatalogues worth a second look.
4. **Bridges between genres.** Points roughly equidistant from two centroids are genre-**blends** (a ritual
   studded with myth, a letter quoting a formula). List the closest few and read them.
5. **Real, or an artefact?** Re-project with **PCA** and **t-SNE** as well as UMAP, at several `n_neighbors`.
   Structure that survives every method and setting is trustworthy; a split that appears only once is probably
   noise — the critical-thinking counterpart to the 3-D view above.
6. **Sub-genres within a genre.** Cluster *only* the myth tablets: do the Baal-cycle texts (1.1–1.6) peel away
   from Kirta / Aqhat? There is structure below the genre level too.